In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import comb, gammaln
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings("ignore")

# ==========================================================
# 1. 数据读取与初步处理
# ==========================================================
data = pd.read_csv("StressLevelDataset.csv")

# 分离自变量与因变量
X = data.drop(columns=["stress_level"]).values
y = data["stress_level"].astype(int).values

# 标准化特征
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
Z = np.column_stack([np.ones(X_scaled.shape[0]), X_scaled])  # 添加截距项

# 训练/测试划分：4:1 比例
Z_train, Z_test, y_train, y_test = train_test_split(Z, y, test_size=0.2, stratify=y, random_state=42)
print(f"Training set: {Z_train.shape}, Test set: {Z_test.shape}")

# ==========================================================
# 2. 定义 Bernstein-Binomial 模型类
# ==========================================================
class BernsteinBinomialRegression:
    def __init__(self, K=3, m=2):
        self.K = K
        self.m = m
        self.alpha = None
        self.phi = None
        self.beta = None
        self.log_likelihood = None
        self.n_params = None

    def _compute_eta(self, Z, alpha, phi, beta):
        n = Z.shape[0]
        Z_beta = Z @ beta
        eta = np.zeros((n, self.K + 1))
        eta[:, 0] = 0 + 1.0 * Z_beta
        for k in range(1, self.K + 1):
            eta[:, k] = alpha[k - 1] + phi[k - 1] * Z_beta
        return eta

    def _compute_lambda(self, eta):
        max_eta = np.max(eta, axis=1, keepdims=True)
        exp_eta = np.exp(eta - max_eta)
        return exp_eta / np.sum(exp_eta, axis=1, keepdims=True)

    def _bernstein_basis(self, p, y_obs):
        return comb(self.m, y_obs) * (p ** y_obs) * ((1 - p) ** (self.m - y_obs))

    def _log_likelihood(self, params, Z, y):
        n, p = Z.shape
        K = self.K
        alpha = params[:K]
        phi = np.exp(params[K:2*K])  # ensure positive
        beta = params[2*K:2*K+p]

        eta = self._compute_eta(Z, alpha, phi, beta)
        lambda_mat = self._compute_lambda(eta)

        logL = 0.0
        for i in range(n):
            mixture_prob = 0.0
            for k in range(K + 1):
                p_val = k / K
                mixture_prob += lambda_mat[i, k] * self._bernstein_basis(p_val, y[i])
            logL += np.log(max(mixture_prob, 1e-12))
        return logL

    def _negative_log_likelihood(self, params, Z, y):
        logL = self._log_likelihood(params, Z, y)
        regularization = 0.001 * np.sum(params ** 2)
        return -(logL / len(y)) + regularization

    def fit(self, Z, y, max_iter=1000):
        n, p = Z.shape
        K = self.K
        np.random.seed(42)
        alpha_init = np.random.normal(0, 0.1, K)
        phi_init = np.log(np.ones(K))
        beta_init = np.random.normal(0, 0.1, p)
        initial_params = np.concatenate([alpha_init, phi_init, beta_init])
        bounds = [(-5, 5)] * len(initial_params)

        result = minimize(
            self._negative_log_likelihood, initial_params,
            args=(Z, y), method="L-BFGS-B", bounds=bounds,
            options={'maxiter': max_iter, 'ftol': 1e-8}
        )

        self.alpha = result.x[:K]
        self.phi = np.exp(result.x[K:2*K])
        self.beta = result.x[2*K:2*K+p]
        self.log_likelihood = self._log_likelihood(result.x, Z, y)
        self.n_params = len(result.x)
        self.optimization_result = result
        return result

    def predict_proba(self, Z):
        eta = self._compute_eta(Z, self.alpha, self.phi, self.beta)
        lambda_mat = self._compute_lambda(eta)
        prob_matrix = np.zeros((Z.shape[0], self.m + 1))
        for i in range(Z.shape[0]):
            for stress_level in range(self.m + 1):
                p_sum = 0
                for k in range(self.K + 1):
                    p_val = k / self.K
                    p_sum += lambda_mat[i, k] * self._bernstein_basis(p_val, stress_level)
                prob_matrix[i, stress_level] = p_sum
        return prob_matrix / np.sum(prob_matrix, axis=1, keepdims=True)

    def predict(self, Z):
        prob_matrix = self.predict_proba(Z)
        return np.argmax(prob_matrix, axis=1)

# ==========================================================
# 3. 5折交叉验证选择最优 K
# ==========================================================
candidate_K = [2, 4, 6, 8, 10]
m = 2
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for K in candidate_K:
    print(f"\nEvaluating K={K} ...")
    accuracies = []
    for train_idx, valid_idx in kf.split(Z_train, y_train):
        Z_tr, Z_val = Z_train[train_idx], Z_train[valid_idx]
        y_tr, y_val = y_train[train_idx], y_train[valid_idx]

        model = BernsteinBinomialRegression(K=K, m=m)
        model.fit(Z_tr, y_tr)
        y_pred = model.predict(Z_val)
        acc = accuracy_score(y_val, y_pred)
        accuracies.append(acc)

    mean_acc = np.mean(accuracies)
    cv_results[K] = mean_acc
    print(f"  Mean CV Accuracy: {mean_acc:.4f}")

# 选择最优 K*
K_star = max(cv_results, key=cv_results.get)
print("\n========================================")
print(f"Optimal Bernstein degree K* = {K_star}")
print("========================================")

# ==========================================================
# 4. 在训练集上用 K* 拟合最终模型，并比较两个模型
# ==========================================================
# Bernstein–Binomial 模型
bernstein_model = BernsteinBinomialRegression(K=K_star, m=m)
bernstein_model.fit(Z_train, y_train)
y_pred_bernstein = bernstein_model.predict(Z_test)
acc_bernstein = accuracy_score(y_test, y_pred_bernstein)

# Beta–Binomial 模型
class BetaBinomialRegression:
    def __init__(self, m=2):
        self.m = m
        self.coefficients = None

    def _log_likelihood(self, params, Z, y):
        logit_p = Z @ params
        p_vals = 1 / (1 + np.exp(-logit_p))
        p_vals = np.clip(p_vals, 1e-5, 1 - 1e-5)
        dispersion = 5.0
        alpha = p_vals * dispersion
        beta = (1 - p_vals) * dispersion
        logL = 0.0
        for i in range(len(y)):
            logL += (
                gammaln(self.m + 1) - gammaln(y[i] + 1) - gammaln(self.m - y[i] + 1)
                + gammaln(y[i] + alpha[i]) + gammaln(self.m - y[i] + beta[i])
                - gammaln(self.m + alpha[i] + beta[i])
                + gammaln(alpha[i] + beta[i]) - gammaln(alpha[i]) - gammaln(beta[i])
            )
        return logL

    def _negative_log_likelihood(self, params, Z, y):
        return -self._log_likelihood(params, Z, y)

    def fit(self, Z, y):
        p = Z.shape[1]
        initial_params = np.zeros(p)
        result = minimize(self._negative_log_likelihood, initial_params, args=(Z, y), method="L-BFGS-B")
        self.coefficients = result.x
        return result

    def predict(self, Z):
        logit_p = Z @ self.coefficients
        p_vals = 1 / (1 + np.exp(-logit_p))
        return np.round(p_vals * self.m).astype(int)

beta_model = BetaBinomialRegression(m=m)
beta_model.fit(Z_train, y_train)
y_pred_beta = beta_model.predict(Z_test)
acc_beta = accuracy_score(y_test, y_pred_beta)

# ==========================================================
# 5. 性能比较
# ==========================================================
print("\n========================================")
print("Performance Comparison on Test Set")
print("========================================")
print(f"Bernstein–Binomial (K={K_star}): Accuracy = {acc_bernstein:.4f}")
print(f"Beta–Binomial: Accuracy = {acc_beta:.4f}")
